# Cross-Dataset Comparison & Domain Gap

Compare Tufts dental (Stage 1 training) vs DC1000 (Stage 2 inference target).
This helps assess the domain gap the YOLO tooth detector must bridge.

In [ ]:
import glob
import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 100

In [ ]:
TUFTS_DIR = "../data/tufts_dental/Radiographs"
DC1000_DIR = "../data/DC1000/train/images"

tufts_files = sorted(glob.glob(os.path.join(TUFTS_DIR, '*.JPG')))
dc1000_files = sorted(glob.glob(os.path.join(DC1000_DIR, '*.png')))

print(f"Tufts: {len(tufts_files)} images")
print(f"DC1000 (train): {len(dc1000_files)} images")

## 1. Side-by-side visual comparison

Compare how the two datasets look — are they similar enough for YOLO transfer?

In [ ]:
rng = np.random.RandomState(42)

fig, axes = plt.subplots(3, 2, figsize=(20, 18))

for row in range(3):
    # Tufts sample
    t_idx = rng.randint(len(tufts_files))
    tufts_img = cv2.imread(tufts_files[t_idx])
    tufts_rgb = cv2.cvtColor(tufts_img, cv2.COLOR_BGR2RGB)
    
    # DC1000 sample
    d_idx = rng.randint(len(dc1000_files))
    dc_img = cv2.imread(dc1000_files[d_idx])
    dc_rgb = cv2.cvtColor(dc_img, cv2.COLOR_BGR2RGB)
    
    axes[row, 0].imshow(tufts_rgb)
    axes[row, 0].set_title(f'Tufts — {os.path.basename(tufts_files[t_idx])} '
                           f'({tufts_img.shape[0]}×{tufts_img.shape[1]})', fontsize=11)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(dc_rgb)
    axes[row, 1].set_title(f'DC1000 — {os.path.basename(dc1000_files[d_idx])} '
                           f'({dc_img.shape[0]}×{dc_img.shape[1]})', fontsize=11)
    axes[row, 1].axis('off')

plt.suptitle('Tufts (YOLO training) vs DC1000 (inference target) — Visual Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Pixel intensity distributions

Compare histogram shapes to see how different the contrast/brightness profiles are.

In [ ]:
def sample_intensity_histogram(file_list, n_samples=50, seed=42):
    """Compute average intensity histogram from a sample of images."""
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(file_list), size=min(n_samples, len(file_list)), replace=False)
    all_hists = []
    for idx in indices:
        img = cv2.imread(file_list[idx], cv2.IMREAD_GRAYSCALE)
        hist, _ = np.histogram(img.ravel(), bins=256, range=(0, 256), density=True)
        all_hists.append(hist)
    return np.mean(all_hists, axis=0)

tufts_hist = sample_intensity_histogram(tufts_files)
dc1000_hist = sample_intensity_histogram(dc1000_files)

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.plot(tufts_hist, label='Tufts dental', color='steelblue', alpha=0.8)
ax.plot(dc1000_hist, label='DC1000', color='coral', alpha=0.8)
ax.set_title('Average Intensity Histogram (grayscale)')
ax.set_xlabel('Pixel intensity')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Mean intensity and contrast per image

In [ ]:
def compute_stats(file_list, n_samples=100, seed=42):
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(file_list), size=min(n_samples, len(file_list)), replace=False)
    means, stds = [], []
    for idx in indices:
        img = cv2.imread(file_list[idx], cv2.IMREAD_GRAYSCALE)
        means.append(img.mean())
        stds.append(img.std())
    return np.array(means), np.array(stds)

t_means, t_stds = compute_stats(tufts_files)
d_means, d_stds = compute_stats(dc1000_files)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(t_means, bins=30, alpha=0.6, color='steelblue', label='Tufts', edgecolor='black')
axes[0].hist(d_means, bins=30, alpha=0.6, color='coral', label='DC1000', edgecolor='black')
axes[0].set_title('Mean Pixel Intensity Distribution')
axes[0].set_xlabel('Mean intensity')
axes[0].legend()

axes[1].hist(t_stds, bins=30, alpha=0.6, color='steelblue', label='Tufts', edgecolor='black')
axes[1].hist(d_stds, bins=30, alpha=0.6, color='coral', label='DC1000', edgecolor='black')
axes[1].set_title('Pixel Std Dev Distribution (contrast)')
axes[1].set_xlabel('Std dev')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Tufts:  mean intensity={t_means.mean():.1f} ± {t_means.std():.1f}, "
      f"mean contrast={t_stds.mean():.1f} ± {t_stds.std():.1f}")
print(f"DC1000: mean intensity={d_means.mean():.1f} ± {d_means.std():.1f}, "
      f"mean contrast={d_stds.mean():.1f} ± {d_stds.std():.1f}")

## 4. What YOLO will see (resized to 640×640)

In [ ]:
YOLO_SIZE = 640

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

rng = np.random.RandomState(123)

for col in range(3):
    # Tufts resized
    t_idx = rng.randint(len(tufts_files))
    t_img = cv2.imread(tufts_files[t_idx])
    t_resized = cv2.resize(t_img, (YOLO_SIZE, YOLO_SIZE))
    t_rgb = cv2.cvtColor(t_resized, cv2.COLOR_BGR2RGB)
    
    # DC1000 resized
    d_idx = rng.randint(len(dc1000_files))
    d_img = cv2.imread(dc1000_files[d_idx])
    d_resized = cv2.resize(d_img, (YOLO_SIZE, YOLO_SIZE))
    d_rgb = cv2.cvtColor(d_resized, cv2.COLOR_BGR2RGB)
    
    axes[0, col].imshow(t_rgb)
    axes[0, col].set_title(f'Tufts @ {YOLO_SIZE}×{YOLO_SIZE}')
    axes[0, col].axis('off')
    
    axes[1, col].imshow(d_rgb)
    axes[1, col].set_title(f'DC1000 @ {YOLO_SIZE}×{YOLO_SIZE}')
    axes[1, col].axis('off')

plt.suptitle('How YOLO sees both datasets (resized to 640×640)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Assessment

Key questions to answer visually:
- Do teeth look structurally similar across datasets?
- Is the contrast/brightness significantly different?
- Are there artifacts in one dataset not present in the other?
- At 640×640, do tooth boundaries remain visible?

In [ ]:
print("Domain Gap Assessment")
print("=" * 50)
print(f"Resolution:     Tufts 840×1615, DC1000 1435×2943")
print(f"Aspect ratio:   Both ~1.9-2.1:1 (good match)")
print(f"Tufts mean px:  {t_means.mean():.1f} ± {t_means.std():.1f}")
print(f"DC1000 mean px: {d_means.mean():.1f} ± {d_means.std():.1f}")
print(f"")
print("Both are panoramic dental radiographs.")
print("YOLO letterpad-resizes to 640×640 with preserved aspect ratio.")
print("Key risk: brightness/contrast differences may need augmentation.")
print("Mitigation: YOLO augmentations (hsv_h, hsv_s, hsv_v) should help.")